In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [2]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [3]:
df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df.drop(columns=["Cabin", "Name", "Ticket", "PassengerId"], inplace=True)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [5]:
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)

df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


In [7]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (712, 8)
Testing data: (179, 8)


In [9]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Original model trained successfully!")

Original model trained successfully!


In [10]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[0 0 0 1 1 1 1 0 1 1]


In [11]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8100558659217877


In [12]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



## Classification Report

The classification report provides Precision, Recall, and F1-score for each class.

- Precision measures how many predicted positives are actually correct.
- Recall measures how many actual positives were correctly identified.
- F1-score combines Precision and Recall into a single performance metric.

These metrics provide a more complete evaluation than accuracy alone.

## Why Accuracy Alone Can Be Misleading

Accuracy only tells us the percentage of correct predictions.

For imbalanced datasets, a model may achieve high accuracy by predicting only the majority class while failing to identify the minority class.

Therefore, Precision, Recall, and F1-score provide a better understanding of model performance.

In [13]:
param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'C': 1, 'solver': 'liblinear'}


In [16]:
print("Tuned Accuracy:", accuracy_score(y_test, y_pred_best))

Tuned Accuracy: 0.7821229050279329


In [14]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_best))

print(classification_report(y_test, y_pred_best))

Accuracy: 0.7821229050279329
              precision    recall  f1-score   support

           0       0.79      0.85      0.82       105
           1       0.76      0.69      0.72        74

    accuracy                           0.78       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



| Model                        | Accuracy                         |
| ---------------------------- | -------------------------------- |
| Original Logistic Regression | *(81.01%)* |
| Tuned Logistic Regression    | *(78.21%)*       |


## Model Comparison

The original Logistic Regression model achieved an accuracy of 81.01%.

After applying GridSearchCV for hyperparameter tuning, the model achieved an accuracy of 78.21%.

In this case, hyperparameter tuning did not improve the model's performance on the test data. This can happen because the best parameters found during cross-validation may not always generalize better to unseen data. Machine learning model performance depends on the dataset, and tuning is not guaranteed to increase accuracy.

In [17]:
print("Best Parameters:", grid.best_params_)

Best Parameters: {'C': 1, 'solver': 'liblinear'}


# Conclusion

In this task, I evaluated the Logistic Regression model using Accuracy, Precision, Recall, and F1-score. I also applied GridSearchCV to tune the model's hyperparameters. The best parameters found were C = 1 and solver = liblinear. Although the tuned model achieved slightly lower accuracy than the original model, this demonstrated that hyperparameter tuning does not always improve performance on unseen data. This task helped me understand proper model evaluation and optimization techniques.